In [1]:
import random
import pickle
import numpy as np
from collections import defaultdict
import scipy.sparse as sp

import os
import random
import pandas as pd
import json
import pickle
import gzip
from tqdm import tqdm

def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)
    
def ReadLineFromFile(path):
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        # Convert bytes to string
        data_str = l.decode('utf-8')
        # Replace 'false' with 'False' and 'true' with 'True'
        data_str = data_str.replace('false', 'False').replace('true', 'True')

        # Parse the JSON string into a dictionary
        yield eval(data_str)
'''
Set seeds
'''
seed = 999
random.seed(seed)
np.random.seed(seed)

In [2]:
DATA_PATH = '../data/'

In [3]:
DATASET = 'beauty'

In [4]:
test_samples = ReadLineFromFile(os.path.join(DATA_PATH, DATASET, 'negative_samples.txt'))
len(test_samples)

22363

In [5]:
sequential_data = ReadLineFromFile(os.path.join(DATA_PATH, DATASET, 'sequential_data.txt'))
item_count = defaultdict(int)
user_items = defaultdict()

for line in sequential_data:
    user, items = line.strip().split(' ', 1)
    items = items.split(' ')
    items = [int(item) for item in items]
    user_items[user] = items
    for item in items:
        item_count[item] += 1

In [6]:
user_items['1']

[1, 2, 3, 4, 5]

In [7]:
all_item = list(item_count.keys())

In [8]:
datamaps = load_json(os.path.join(DATA_PATH, DATASET, 'datamaps.json'))
user2id = datamaps['user2id']
item2id = datamaps['item2id']
user_list = list(datamaps['user2id'].keys())
item_list = list(datamaps['item2id'].keys())
id2item = datamaps['id2item']
id2user = datamaps['id2user']

In [9]:
list(id2user.keys())[:4]

['1', '2', '3', '4']

In [ ]:
train_negative = []
for user in tqdm(list(id2user.keys())):
    user_seq = user_items[user][:-1]
    user_seq = set([str(x) for x in user_seq])
    candidate_samples = []
    candidate_num = len(user_seq)
    already_samples = test_samples[int(user)-1].split(' ', 1)[1].split(' ')
    already_samples = set([str(x) for x in already_samples])
    while len(candidate_samples) < candidate_num:
        sample_ids = np.random.choice(all_item, candidate_num, replace=False)
        sample_ids = [str(item) for item in sample_ids if item not in user_seq and item not in candidate_samples and item not in already_samples]
        candidate_samples.extend(sample_ids)
    candidate_samples = candidate_samples[:candidate_num]
    train_negative.append([user] + candidate_samples)
    

In [11]:
train_negative[0], user_items['1'], test_samples[0]

(['1', '622', '3642', '1695', '403'],
 [1, 2, 3, 4, 5],
 '1 3408 9109 8050 1546 6421 6705 5011 8699 11168 3347 4394 1847 1037 3112 10437 5215 10080 11503 9141 5090 5208 9072 11691 920 1431 7862 1044 5821 2936 1518 7872 10716 11935 11613 5953 5345 10764 6747 3105 1890 3453 8226 6515 1306 7779 5594 11186 3626 10965 9331 12066 10343 1947 11245 11167 4832 3706 4421 3047 102 11455 9586 7059 6871 5987 6916 5871 5644 2376 2682 7989 3689 7582 897 5454 4357 8464 6752 11637 2854 8434 10267 8187 5254 3684 9090 9863 2387 992 1047 1544 435 9166 2225 6010 5549 5360 5477 1117')

In [12]:
len(train_negative)

22363

In [13]:
save_pickle(train_negative,os.path.join(DATA_PATH,DATASET,'train-negatives.pkl'))

# Check overlaps

In [10]:
train_negative = load_pickle(os.path.join(DATA_PATH,DATASET,'train-negatives.pkl'))
len(train_negative)

22363

In [11]:
test_negs = []
for idx in range(len(test_samples)):
    lst = test_samples[idx].split(' ')[1:]
    user = test_samples[idx].split(' ')[0]
    for val in lst:
        test_negs.append((int(user) - 1, int(val)))
len(test_negs)

2213937

In [12]:
test_negs[:10]

[(0, 3408),
 (0, 9109),
 (0, 8050),
 (0, 1546),
 (0, 6421),
 (0, 6705),
 (0, 5011),
 (0, 8699),
 (0, 11168),
 (0, 3347)]

In [13]:
train_negs = []
for idx in range(len(train_negative)):
    lst = train_negative[idx]
    user = lst[0]
    items = lst[1:]
    for item in items:
        train_negs.append((int(user)-1,int(item)))

In [14]:
len(train_negs)

175908

In [15]:
train_negs[:10]

[(0, 622),
 (0, 3642),
 (0, 1695),
 (0, 403),
 (1, 2392),
 (1, 8511),
 (1, 5833),
 (1, 170),
 (1, 4654),
 (1, 7379)]

In [16]:
inter = []
for user,items in user_items.items():
    new_user = int(user) - 1
    for item in items:
        item = int(item)
        inter.append((new_user,item))

In [17]:
len(inter)

198502

In [18]:
inter[:10]

[(0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (1, 6),
 (1, 7),
 (1, 8),
 (1, 9),
 (1, 10)]

In [19]:
len(set(train_negs))

175908

In [20]:
len(set(test_negs))

2213937

In [21]:
len(set(inter))

198502

In [22]:
common1 = set(train_negs).intersection(set(test_negs))
len(common1)

1405

In [23]:
common2 = set(train_negs).intersection(set(inter))
len(common2)

20

In [24]:
common2

{(160, 1584),
 (1367, 6962),
 (1574, 7343),
 (1582, 7151),
 (2206, 4629),
 (2652, 7235),
 (5190, 1823),
 (5370, 8756),
 (5883, 1445),
 (6314, 7335),
 (7003, 5055),
 (7803, 10446),
 (9665, 1887),
 (10881, 9968),
 (13343, 7508),
 (14515, 5213),
 (17645, 4876),
 (17777, 7967),
 (21056, 296),
 (21987, 11278)}

In [28]:
print("Common 1: %:",1405 * 100.0 / 175908,'%')

Common 1: %: 0.7987129635946063 %


In [27]:
print("Common 2: %:",20 * 100.0 / 175908,'%')

Common 2: %: 0.011369579552948131 %
